
# NB_OTT_BuildIncidentGroups

Objetivo:
Agregar los tickets previamente enriquecidos con su cluster_id y construir una representación operacional de cada grupo detectado.

 Entrada:
    Gold.ClusteredTickets

 Salida:
   Gold.IncidentGroups

 Este notebook forma parte del pipeline operacional.

 IMPORTANTE:
 - No realiza evaluación del clustering.
 - No utiliza incident_id como parte de la lógica productiva.
 - El ground truth se analiza posteriormente en notebooks específicos de evaluación.


In [ ]:
from pyspark.sql import functions as F

In [ ]:
# ============================================================
# 1. Carga de tickets enriquecidos
#
# Gold.ClusteredTickets contiene los tickets Silver junto con el cluster_id asignado por DBSCAN.
# ============================================================

df_clustered = spark.table(
    "Gold.ClusteredTickets"
)

clustered_count = df_clustered.count()

assert clustered_count > 0, (
    "Gold.ClusteredTickets does not contain any records."
)

print(
    f"Clustered tickets loaded: {clustered_count}"
)

In [ ]:
# ============================================================
# 2. Construcción de grupos operacionales
#
# Cada cluster se transforma en una entidad agregada que resume:
#
# - número de tickets,
# - ventana temporal,
# - servicios afectados,
# - plataformas,
# - dispositivos,
# - países,
# - códigos de error,
# - categorías,
# - prioridades,
# - tickets que forman el grupo.
#
# Esta representación será posteriormente consumida por GenAI.
# ============================================================

df_incident_groups = (
    df_clustered
    .groupBy(
        "cluster_id"
    )
    .agg(
        F.count("*")
        .alias(
            "ticket_count"
        ),

        F.min(
            "created_at"
        )
        .alias(
            "first_ticket_at"
        ),

        F.max(
            "created_at"
        )
        .alias(
            "last_ticket_at"
        ),

        F.collect_set(
            "service"
        )
        .alias(
            "services"
        ),

        F.collect_set(
            "platform"
        )
        .alias(
            "platforms"
        ),

        F.collect_set(
            "device_type"
        )
        .alias(
            "device_types"
        ),

        F.collect_set(
            "country"
        )
        .alias(
            "countries"
        ),

        F.collect_set(
            "error_code"
        )
        .alias(
            "error_codes"
        ),

        F.collect_set(
            "category"
        )
        .alias(
            "categories"
        ),

        F.collect_set(
            "priority"
        )
        .alias(
            "priorities"
        ),

        F.collect_list(
            "ticket_id"
        )
        .alias(
            "ticket_ids"
        )
    )
)

In [ ]:
# ============================================================
# 3. Duración temporal del grupo
#
# Calcula el intervalo entre el primer y el último ticket del cluster. Se utiliza como información descriptiva del grupo y no como duración real del incidente.
# ============================================================

df_incident_groups = (
    df_incident_groups
    .withColumn(
        "duration_minutes",
        (
            F.col(
                "last_ticket_at"
            ).cast("long")
            -
            F.col(
                "first_ticket_at"
            ).cast("long")
        ) / 60.0
    )
)

In [ ]:
# ============================================================
# 4. Preparación del contexto textual
#
# Para cada cluster se construye una colección de textos combinando título y descripción.
#
# Estos textos serán posteriormente muestreados por NB_OTT_GenerateIncidentSummaries.
# ============================================================

df_cluster_texts = (
    df_clustered
    .groupBy(
        "cluster_id"
    )
    .agg(
        F.collect_list(
            F.concat_ws(
                " | ",
                F.coalesce(
                    F.col("title"),
                    F.lit("")
                ),
                F.coalesce(
                    F.col("description"),
                    F.lit("")
                )
            )
        )
        .alias(
            "ticket_texts"
        )
    )
)

In [ ]:
# ============================================================
# 5. Construcción final de IncidentGroups
# ============================================================

df_incident_groups = (
    df_incident_groups
    .join(
        df_cluster_texts,
        on="cluster_id",
        how="left"
    )
    .withColumn(
        "_processed_at",
        F.current_timestamp()
    )
)

In [ ]:
# ============================================================
# 6. Verificación de calidad de los datos
#
# Se verifica: existencia de grupos, un registro por cluster, ausencia de grupos vacíos y existencia de contexto textual.
# ============================================================

group_count = (
    df_incident_groups.count()
)

distinct_clusters = (
    df_incident_groups
    .select(
        "cluster_id"
    )
    .distinct()
    .count()
)

empty_groups = (
    df_incident_groups
    .filter(
        F.col(
            "ticket_count"
        ) <= 0
    )
    .count()
)

missing_ticket_texts = (
    df_incident_groups
    .filter(
        F.col(
            "ticket_texts"
        ).isNull()
        |
        (
            F.size(
                F.col(
                    "ticket_texts"
                )
            ) == 0
        )
    )
    .count()
)

assert group_count > 0

assert group_count == distinct_clusters, (
    "More than one row found for the same cluster_id."
)

assert empty_groups == 0, (
    f"Found {empty_groups} empty incident groups."
)

assert missing_ticket_texts == 0, (
    f"Found {missing_ticket_texts} groups without ticket texts."
)


print("Incident group validation")
print("-------------------------")
print(
    "Groups:",
    group_count
)
print(
    "Distinct cluster IDs:",
    distinct_clusters
)
print(
    "Empty groups:",
    empty_groups
)

In [ ]:
# ============================================================
# 7. Persistencia en Gold
#
# Para el proyecto, la tabla se reconstruye mediante overwrite. El diseño puede evolucionar posteriormente hacia una estrategia incremental.
# ============================================================

spark.sql(
    "CREATE SCHEMA IF NOT EXISTS Gold"
)

(
    df_incident_groups.write
    .format("delta")
    .mode("overwrite")
    .option(
        "overwriteSchema",
        "true"
    )
    .saveAsTable(
        "Gold.IncidentGroups"
    )
)

print(
    "Gold.IncidentGroups written successfully."
)

In [ ]:
# ============================================================
# 8. Validación posterior a persistencia
# ============================================================

df_saved = spark.table(
    "Gold.IncidentGroups"
)

saved_count = (
    df_saved.count()
)

assert saved_count == group_count, (
    f"Expected {group_count} groups "
    f"but persisted {saved_count}."
)

print("BuildIncidentGroups completed")
print("-----------------------------")
print(
    "Clustered tickets:",
    clustered_count
)
print(
    "Incident groups:",
    saved_count
)

print(
    "\nNB_OTT_BuildIncidentGroups "
    "completed successfully."
)